# 🎬 Dailies Jr. Studio
### ISOM 260 · Your team builds the harness. The professor rolls the camera.

Wednesday you watched **Dailies Jr.** turn one idea into a 20-second video: a 🎬 Director, one 🧑 character sheet, a ✅ Continuity Checker in code, and an ✂️ editor in code. Same model, different harness.

**Today your team runs the studio.** You pick an industry and an idea, run the harness, and then **make it yours**: add a rule, an agent, a voice, a wall. Everything here runs on your **class pass** and costs about a cent. You get a free **storyboard video** of your plan.

At the end, every team prints a **🎟️ Premiere ticket**. The professor renders the best ones as real AI video on the big screen, and **the class votes.**

| Your harness needs | You'll build it in |
|---|---|
| 📜 one rule written down | Upgrade B or D |
| 🧱 one check in code | Upgrade A, C or D |
| 🧑 one human gate | Step 6 (you approve before the ticket prints) |

*That's also exactly what your HW #9 pitch asks for.*

### 🧭 How to use this notebook
- ▶️ **Run the cells in order.** Click a cell, press **Shift + Enter**.
- ✏️ **Only cells marked ✏️ need typing.** Everything else: just run it.
- 👥 **One laptop per team drives**, everyone else argues. Teams of 2 or 3.

## 🚀 Step 1: Setup (2 minutes)
Same class pass as always: 🔑 Secrets → `GOOGLE_API_KEY` = your class pass (like `CODE:Maya R`), **Notebook access on**. When Colab asks to **Grant access**, click it right away (if you wait, the cell times out; just run it again).

In [ ]:
!pip install -q -U "google-genai>=2.34,<3"

In [ ]:
# @title ▶️ Run me: connect to Gemini (no typing needed)
from google.colab import userdata
from google import genai
from google.genai import types, errors
import json, re, math, time, textwrap, html
from IPython.display import display, HTML, Markdown

class Stop(Exception):
    """A friendly stop: shows one clear message instead of a wall of red."""
    def _render_traceback_(self): return ["⛔ " + str(self)]

def help_box(title, steps):
    display(HTML("<div style='border-left:5px solid #b42318;background:#fff5f5;color:#1d2226;padding:12px 16px;border-radius:6px;"
                 "font-family:Arial,sans-serif;font-size:14px;max-width:640px'><b>😵 " + title + "</b><ol style='margin:8px 0 0 18px'>"
                 + "".join(f"<li>{s}</li>" for s in steps) + "</ol></div>"))

try:
    GOOGLE_API_KEY = (userdata.get("GOOGLE_API_KEY") or "").strip()   # your class pass (CODE:Your Name), or your own Gemini key
except Exception as e:
    help_box("Colab can't see your class pass yet", [
        "Click the 🔑 key icon in the left sidebar.",
        "Find (or add) the secret named exactly <code>GOOGLE_API_KEY</code>. Its value is your class pass, like <code>CODE:Maya R</code>.",
        "Switch on <b>Notebook access</b> for it. Every new notebook needs this switch.",
        "Run this cell again."])
    raise Stop(f"No class pass found ({type(e).__name__}). Follow the steps above.") from None

if ":" not in GOOGLE_API_KEY and not GOOGLE_API_KEY.startswith("AIza"):
    help_box("That doesn't look like a class pass", [
        "A class pass has a colon in the middle: the class code, a colon, your first name and last initial. Like <code>CODE:Maya R</code>.",
        "Fix the value in 🔑 Secrets, then run this cell again."])
    raise Stop("The class pass needs a colon. Fix it in Secrets and run again.")
PASS_NAME = GOOGLE_API_KEY.split(":", 1)[1].strip().split(" ")[0].title() if ":" in GOOGLE_API_KEY else ""

# A class pass goes through the Sawyer AI Sandbox: no personal key, no credit card. A personal key goes straight to Google.
SANDBOX = {"base_url": "https://sawyer-ai-sandbox.vercel.app/api/gemini"} if ":" in GOOGLE_API_KEY else None
client = genai.Client(api_key=GOOGLE_API_KEY, http_options={**(SANDBOX or {}), "timeout": 60000})   # never wait more than a minute
MODEL  = "gemini-2.5-flash"      # a brand-new key may be told 2.5 is "not found" → use "gemini-3.8-flash" (also free)
FAST   = types.ThinkingConfig(thinking_budget=0) if MODEL.startswith("gemini-2.5-flash") else None   # writing and checking skip hidden "thinking": faster, cheaper

FLASH  = (0.30, 2.50)      # $ per 1M tokens in / out, Gemini 2.5 Flash paid tier
ARGON  = (2.00, 10.00)     # Gemini 4 Argon, intro price to Nov 30
BILL = {"calls": 0, "in": 0, "out": 0, "thoughts": 0}

def add_to_bill(u):
    BILL["calls"] += 1
    BILL["in"] += (u.prompt_token_count or 0) if u else 0
    BILL["out"] += (u.candidates_token_count or 0) if u else 0
    BILL["thoughts"] += (u.thoughts_token_count or 0) if u else 0

def since(before):            # the part of the bill spent since `before = dict(BILL)`
    return {k: BILL[k] - before[k] for k in BILL}

def dollars(b, price=FLASH):
    return (b["in"] * price[0] + (b["out"] + b["thoughts"]) * price[1]) / 1e6

def bill_line(b):
    return f"{b['calls']:g} AI calls · ≈ ${dollars(b):.4f} on Flash · ${dollars(b, ARGON):.4f} on Argon"

def call_gemini(contents, config, retries=3):
    """Every AI call in this notebook goes through here, so the bill is always complete and hiccups get a retry."""
    for attempt in range(retries):
        try:
            r = client.models.generate_content(model=MODEL, contents=contents, config=config)
            add_to_bill(r.usage_metadata)
            return r
        except errors.APIError as e:
            msg = (getattr(e, "message", None) or str(e))
            if e.code == 404:
                raise Stop(f"Gemini can't find the model '{MODEL}'. In Part 1, set MODEL = \"gemini-3.8-flash\" and run from Part 1 again.") from None
            if e.code in (400, 401, 403) or any(w in msg.lower() for w in ("budget", "cap reached", "class pass", "daily")):
                raise Stop(msg) from None    # a pass, budget or request problem: retrying won't help
            wait = 8 * (attempt + 1)
            print(f"   (Gemini said {e.code}: waiting {wait}s and trying again…)")
            time.sleep(wait)
    raise Stop("No answer after 3 tries. Wait a minute and run the cell again.")

def read_json(text):
    """Models wrap JSON in ``` fences or add a sentence. Pull out the JSON, or return None."""
    text = text or ""
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    if fence: text = fence.group(1)
    starts = [i for i in (text.find("{"), text.find("[")) if i != -1]
    if not starts: return None
    try: return json.loads(text[min(starts): max(text.rfind("}"), text.rfind("]")) + 1])
    except json.JSONDecodeError: return None

def ask(prompt, as_json=False):
    """One AI call with no tools. as_json=True insists on JSON and asks again if the handoff is broken."""
    config = types.GenerateContentConfig(response_mime_type="application/json" if as_json else None, thinking_config=FAST)
    for _ in range(3):
        r = call_gemini(prompt, config)
        if not as_json: return r.text or ""
        data = read_json(r.text)
        if isinstance(data, dict): return data
        print("   (that handoff was not valid JSON, asking again…)")
    return {}

try:   # one tiny real call, so a problem shows up now and not in the middle of the build
    client.models.generate_content(model=MODEL, contents="Reply with one word: ready", config=types.GenerateContentConfig(thinking_config=FAST))
except Exception as e:
    help_box("Gemini said no", [html.escape(getattr(e, "message", None) or str(e)),
        "Check your pass in 🔑 Secrets: <code>CODE:First L</code>, the code from the board.",
        "Test it at <b>sawyer-ai-sandbox.vercel.app/start</b>. That page says exactly what's wrong.",
        "Still stuck? Raise your hand."])
    raise Stop("Connection test failed. Read the box above.") from None

hi = f" Hi, {PASS_NAME}! 👋" if PASS_NAME else ""
print(("✅ Connected through the Sawyer AI Sandbox." if SANDBOX else "✅ Connected with your own Gemini key.") + hi + " On to Part 2.")

In [ ]:
# @title ▶️ Run me: the studio (Director, Checker, storyboard editor). No typing needed.
import os, io, subprocess, base64
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Video

STEADY = types.GenerateContentConfig(response_mime_type="application/json", thinking_config=FAST, temperature=0.0)
WARM   = types.GenerateContentConfig(response_mime_type="application/json", thinking_config=FAST, temperature=0.7)
def as_json(prompt, cfg=STEADY):
    for _ in range(3):
        data = read_json(call_gemini(prompt, cfg).text)
        if isinstance(data, dict): return data
        print("   (that handoff was not valid JSON, asking again…)")
    raise Stop("The model didn't return valid JSON three times. Run the cell again.")

def card(title, body, color="#0a66c2"):
    display(HTML(f"""<div style="max-width:760px;border:1px solid #d0d7de;border-left:5px solid {color};border-radius:8px;padding:10px 14px;margin:8px 0;
        background:#fff;color:#1d2226;font-family:-apple-system,Segoe UI,Arial,sans-serif;font-size:14px;line-height:1.5">
        <div style="font-size:12px;color:#57606a;margin-bottom:4px">{title}</div>{body}</div>"""))
def esc(s): return html.escape(str(s))
def todo(s): return (not isinstance(s, str)) or s.strip() == "" or s.strip().upper().startswith("TODO")
GREEN, RED, AMBER, BLUE, GREY = "#1a7f37", "#b42318", "#9a6700", "#0a66c2", "#57606a"
CAMERAS = ("slow push in", "slow pan left", "slow pan right", "static")

# ── 🎬 the Director (a model) ──
def direct(idea, industry, notes=""):
    return as_json(f"""You are the director of a 20-second 16:9 social video for the {industry} industry. Idea: {idea}
Write the plan as ONLY JSON:
{{"title": "...",
  "brand": "the business or product name in the video (invented, never a real company)",
  "character": {{"name": "...", "look": "ONE sentence, very specific and visual: age, hair, face, exact clothes and colors, one signature item"}},
  "style": "ONE sentence: art style, lighting, color palette, lens",
  "shots": [{{"n": 1, "seconds": 5, "action": "what we see, one sentence", "camera": "slow push in | slow pan left | slow pan right | static", "caption": "max 6 words", "narration": ""}}]}}
Exactly 4 shots, 16 to 24 seconds in total. The character appears in every shot. No real brands, logos or real people.{notes}""", WARM)

# ── ✅ the base Continuity Checker (code, from Wednesday) ──
BANNED = ["logo", "starbucks", "dunkin", "nike", "apple", "amazon", "google", "taylor swift", "celebrity"]
def base_check(p):
    problems, shots = [], p.get("shots", [])
    if len(shots) != 4: problems.append(f"needs exactly 4 shots, got {len(shots)}")
    total = sum(float(s.get("seconds", 0) or 0) for s in shots)
    if not 16 <= total <= 24: problems.append(f"total length {total:g}s, must be 16–24s")
    for s in shots:
        if float(s.get("seconds", 0) or 0) > 6: problems.append(f"shot {s.get('n')} is {s.get('seconds')}s, max 6")
        if len(str(s.get("caption", "")).split()) > 6: problems.append(f"shot {s.get('n')} caption is longer than 6 words")
        if s.get("camera") not in CAMERAS: problems.append(f"shot {s.get('n')} camera '{s.get('camera')}' isn't one our editor can do")
    text = json.dumps(p).lower()
    problems += [f"banned word: '{w}'" for w in BANNED if re.search(rf"\b{re.escape(w)}\b", text)]
    if len(p.get("character", {}).get("look", "").split()) < 12: problems.append("character look is too vague (under 12 words)")
    return problems

def show_plan(p, title, color=BLUE):
    rows = "".join(f"<li>[{esc(s.get('seconds'))}s · {esc(s.get('camera'))}] {esc(s.get('action'))} <i>“{esc(s.get('caption'))}”</i>"
                   + (f"<br><span style='color:#57606a'>🎙️ {esc(s['narration'])}</span>" if s.get("narration") else "") + "</li>" for s in p.get("shots", []))
    card(title, f"<b>{esc(p.get('title'))}</b> · {esc(p.get('brand', ''))}<br>🧑 <b>{esc(p['character']['name'])}</b>: {esc(p['character']['look'])}"
                f"<br>🎨 {esc(p.get('style'))}<ol style='margin:6px 0 0 18px'>{rows}</ol>", color)

# ── ✂️ the storyboard editor (code, free) ──
W, H, FPS = 1280, 720, 25
os.makedirs("board", exist_ok=True)
def font(size):
    for f in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"]:
        if os.path.exists(f): return ImageFont.truetype(f, size)
    return ImageFont.load_default()
PALETTE = [((28, 36, 64), (88, 52, 110)), ((20, 60, 70), (40, 110, 120)), ((70, 40, 30), (150, 90, 50)), ((30, 50, 30), (70, 120, 70))]
def board_frame(p, s, i):
    a, b = PALETTE[i % 4]
    im = Image.new("RGB", (W, H)); d = ImageDraw.Draw(im, "RGBA")
    for y in range(H):
        t = y / H; d.line([(0, y), (W, y)], fill=tuple(int(a[k] + (b[k] - a[k]) * t) for k in range(3)))
    d.text((48, 36), f"{p.get('title', '')}  ·  SHOT {s.get('n')}  ·  {s.get('seconds')}s  ·  {s.get('camera')}", font=font(26), fill=(255, 255, 255, 190))
    y = 170
    for line in textwrap.wrap(str(s.get("action", "")), 38)[:5]:
        d.text((80, y), line, font=font(48), fill="white"); y += 64
    d.text((80, y + 16), f"STARRING {p['character']['name'].upper()}", font=font(28), fill=(255, 255, 255, 200))
    cap = str(s.get("caption", "")); f = font(50); w = d.textlength(cap, font=f)
    d.rounded_rectangle([(W - w) / 2 - 26, H - 190, (W + w) / 2 + 26, H - 108], 16, fill=(0, 0, 0, 160))
    d.text(((W - w) / 2, H - 180), cap, font=f, fill="white")
    if s.get("narration"):
        f2 = font(24)
        for k, nar in enumerate(textwrap.wrap(str(s["narration"]), 80)[:2]):
            w2 = d.textlength(nar, font=f2)
            d.text(((W - w2) / 2, H - 92 + k * 32), nar, font=f2, fill=(255, 235, 160))
    return im

def storyboard(p, name):
    clips = []
    for i, s in enumerate(p["shots"]):
        png, mp4 = f"board/{name}_{i+1}.png", f"board/{name}_{i+1}.mp4"
        board_frame(p, s, i).save(png)
        sec = max(1.5, float(s.get("seconds", 5) or 5))
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-loop", "1", "-i", png, "-vf",
                        f"scale=2560:-1,zoompan=z='min(zoom+0.0010,1.12)':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={int(sec*FPS)}:s={W}x{H}:fps={FPS},format=yuv420p",
                        "-t", str(sec), "-c:v", "libx264", "-preset", "veryfast", mp4], check=True)
        clips.append((mp4, sec))
    cmd, filt, prev, off = ["ffmpeg", "-y", "-loglevel", "error"], "", "0:v", 0.0
    for c, _ in clips: cmd += ["-i", c]
    for i in range(1, len(clips)):
        off += clips[i - 1][1] - 0.4
        filt += f"[{prev}][{i}:v]xfade=transition=fade:duration=0.4:offset={off:.2f}[v{i}];"; prev = f"v{i}"
    out = f"{name}.mp4"
    subprocess.run(cmd + ["-filter_complex", filt.rstrip(";"), "-map", f"[{prev}]", "-c:v", "libx264", "-preset", "veryfast", "-pix_fmt", "yuv420p", out], check=True)
    display(Video(out, embed=True, width=720))
    return out
print("✅ The studio is open. On to Step 2.")

## 🏭 Step 2: Your pitch (3 minutes)
Pick the industry you want to work in (Session 9!) and one line of idea. A 20-second ad, a listing tour, a recruiting teaser, a menu reveal: anything with **one character** and **one business**.

In [ ]:
# ✏️ Your team. Pick an industry, write one line, then run.
TEAM     = "TODO: your team name"
INDUSTRY = "Marketing"  # @param ["Marketing", "Finance", "Real estate", "Restaurant", "Healthcare", "HR / Recruiting", "Sports", "Nonprofit"]
IDEA     = "TODO: one line, e.g. a 20-second ad for a late-night ramen truck near the Common"

if todo(TEAM) or todo(IDEA):
    print("✏️  Fill in TEAM and IDEA, then run this cell again.")
else:
    card(f"🎬 {esc(TEAM)} · {esc(INDUSTRY)}", f"<b>The idea:</b> {esc(IDEA)}", BLUE)

## 🎞️ Step 3: Run Wednesday's harness as it is (3 minutes)
The Director plans, the base Checker checks, and the editor cuts a **storyboard video**: the plan as moving cards. It's free and instant. This is **v1**, the one you're about to beat.

In [ ]:
# @title ▶️ Run me: v1 = Wednesday's harness
if todo(TEAM) or todo(IDEA): raise Stop("Fill in Step 2 first.")
before = dict(BILL)
V1 = direct(IDEA, INDUSTRY)
p1 = base_check(V1)
if p1:
    print("⛔ Base Checker FAIL: " + "; ".join(p1) + "\n🔁 Back to the Director once…")
    V1 = direct(IDEA, INDUSTRY, "\nFix these problems from the last plan: " + "; ".join(p1))
show_plan(V1, "🎞️ v1 plan (Wednesday's harness)")
storyboard(V1, "v1")
print(f"💵 {bill_line(since(before))}. The editor and the Checker cost $0.")

## 🛠️ Step 4: Make it yours. Pick at least TWO upgrades (15 minutes)
Each upgrade is one ✏️ cell. **Turn it on, edit it, run it.** Then Step 5 runs your whole harness. Name the reason for each one, like Wednesday: a split with no reason counts as wrong.

| Upgrade | Kind | The reason it exists |
|---|---|---|
| 🧱 **A. Your own rule in code** | code | can't be talked around |
| 🛡️ **B. A Brand Guardian agent** (never writes) | 📜 a model + its rules | specialization |
| 🎙️ **C. A Narrator + a timing wall** | a model + 🧱 code | the voice must fit the seconds |
| ⚖️ **D. Your industry's red lines** | 🧱 code | what this industry must never say |

In [ ]:
# ✏️ UPGRADE A · your own Checker rule, in code. Edit the function, keep the name. Return a list of problems ([] means PASS).
USE_A = True

def my_rule(plan):
    problems = []
    last = plan["shots"][-1]
    if plan.get("brand", "").lower() not in str(last.get("caption", "")).lower():
        problems.append("the last shot's caption must name the brand")      # ← change this rule to your own idea
    return problems
# Ideas: the first shot must be 4s or less (hook fast) · every caption ends without "!" · no two shots in a row use the same camera move

In [ ]:
# ✏️ UPGRADE B · a Brand Guardian. It never writes. Give it your brand's rules, one per line.
USE_B = True
BRAND_RULES = """
The tone is warm and confident, never pushy.
Never promise a result the business can't guarantee.
Show the product being used, not just held.
"""

In [ ]:
# ✏️ UPGRADE C · a Narrator writes a voice-over line per shot, and code checks it fits the time.
USE_C = True
VOICE = "a friendly, upbeat local narrator"   # who is talking?
WORDS_PER_SECOND = 2.5                         # how fast people actually speak in an ad

In [ ]:
# ✏️ UPGRADE D · your industry's red lines: phrases this industry must never say. Pre-filled; add your own.
USE_D = True
RED_LINES = {
 "Finance":        ["guaranteed return", "risk-free", "can't lose", "get rich"],
 "Healthcare":     ["cure", "guaranteed results", "no side effects", "doctor recommended"],
 "Real estate":    ["perfect for young families", "exclusive neighborhood", "no kids", "ideal for singles"],
 "HR / Recruiting":["young and energetic", "digital native", "recent grads only", "culture fit"],
 "Restaurant":     ["healthiest", "zero calories", "cures"],
 "Marketing":      ["best in the world", "#1", "100% guaranteed"],
 "Sports":         ["guaranteed to win", "no injuries"],
 "Nonprofit":      ["100% of your donation", "guaranteed impact"],
}
MY_RED_LINES = []   # e.g. ["limited time only"]

## 🏗️ Step 5: Run YOUR harness (5 minutes)
Same Director, same idea. Now every upgrade you turned on gets a say, every failure goes back to the Director **once** with the reasons, and the report card shows **which layer caught what**.

In [ ]:
# @title ▶️ Run me: v2 = your harness
def guardian(p):
    j = as_json("You are a Brand Guardian. You never rewrite anything. Check this video plan against the brand rules. "
                'Reply with ONLY JSON: {"verdict": "PASS" or "FAIL", "problems": ["each broken rule, quoted, with the shot number"]}'
                f"\n\nBRAND RULES:\n{BRAND_RULES}\n\nPLAN:\n{json.dumps(p)}")
    return [str(x) for x in (j.get("problems") or [])] if str(j.get("verdict", "")).upper().startswith("FAIL") else []

def narrate(p):
    j = as_json(f"Write a voice-over for this 4-shot video, spoken by {VOICE}. One line per shot. Each line must be speakable in the shot's seconds "
                f"at {WORDS_PER_SECOND} words per second. "
                'Reply with ONLY JSON: {"lines": ["shot 1 line", "shot 2 line", "shot 3 line", "shot 4 line"]}'
                f"\n\nPLAN:\n{json.dumps(p)}", WARM)
    for s, line in zip(p["shots"], j.get("lines", [])): s["narration"] = str(line)
    return p

def timing_wall(p):
    out = []
    for s in p["shots"]:
        n, cap = len(str(s.get("narration", "")).split()), int(float(s.get("seconds", 0) or 0) * WORDS_PER_SECOND)
        if n > cap: out.append(f"shot {s['n']} narration is {n} words, only {cap} fit in {s['seconds']}s")
    return out

def red_line_wall(p):
    lines = RED_LINES.get(INDUSTRY, []) + list(MY_RED_LINES)
    text = json.dumps(p).lower()
    return [f"red line for {INDUSTRY}: '{w}'" for w in lines if w.lower() in text]

def layers(p):
    found = [("🧱 base Checker", x) for x in base_check(p)]
    if USE_A:
        try: found += [("🧱 your rule (A)", x) for x in my_rule(p)]
        except Exception as e: found.append(("🧱 your rule (A)", f"your rule crashed: {e}. Fix my_rule() in Upgrade A."))
    if USE_D: found += [("🧱 red lines (D)", x) for x in red_line_wall(p)]
    if USE_C: found += [("🧱 timing wall (C)", x) for x in timing_wall(p)]
    if USE_B: found += [("📜 Brand Guardian (B)", x) for x in guardian(p)]
    return found

before = dict(BILL)
V2 = json.loads(json.dumps(V1))
if USE_C: V2 = narrate(V2)
first = layers(V2)
if first:
    notes = "\nThe last plan was rejected. Fix ONLY these problems and keep everything else:\n- " + "\n- ".join(x for _, x in first)
    if MY_RED_LINES or USE_D: notes += "\nNever use these phrases: " + ", ".join(RED_LINES.get(INDUSTRY, []) + list(MY_RED_LINES))
    V2 = direct(IDEA, INDUSTRY, notes + "\n\nLAST PLAN:\n" + json.dumps(V2))
    if USE_C: V2 = narrate(V2)
final = layers(V2)
rows = "".join(f"<tr><td style='padding:3px 8px'>{esc(l)}</td><td style='padding:3px 8px'>{esc(x)}</td></tr>" for l, x in first) or "<tr><td>nothing</td></tr>"
card(f"🧾 Report card: what your layers caught on the first try ({len(first)})",
     f"<table style='border-collapse:collapse;font-size:13px'>{rows}</table><div style='margin-top:8px'>"
     + ("✅ <b>After one round back to the Director: all layers PASS.</b>" if not final else
        "🟡 <b>Still open after one round:</b> " + esc("; ".join(x for _, x in final)) + " (Step 6 decides: ship it or not?)") + "</div>",
     GREEN if not final else AMBER)
show_plan(V2, "🏗️ v2 plan (your harness)", GREEN if not final else AMBER)
storyboard(V2, "v2")
print(f"💵 Your harness: {bill_line(since(before))}. Every 🧱 layer was free.")

## 🧑 Step 6: The human gate (2 minutes)
Your team is the register. **Nothing premieres until a person says yes.** Watch v2, read the report card, then decide. If something's still open, say why you'd ship it anyway, or say no.

In [ ]:
# ✏️ Your team's decision. "yes, because ..." or "no, because ...". Then run.
APPROVE = "TODO: yes, because ... / no, because ..."

if todo(APPROVE):
    print("✏️  A person must decide. Write yes or no, and why, then run.")
elif not APPROVE.strip().lower().startswith("yes"):
    card("🧑 Human gate: NOT approved", f"{esc(APPROVE)}<br>Good: a gate that never says no isn't a gate. Fix the plan (Step 4 → Step 5) and try again.", AMBER)
else:
    TICKET = {"team": TEAM, "industry": INDUSTRY, "idea": IDEA, "approved": APPROVE,
              "upgrades": [n for n, on in [("A rule", USE_A), ("B guardian", USE_B), ("C narrator", USE_C), ("D red lines", USE_D)] if on],
              "plan": V2}
    line = json.dumps(TICKET, ensure_ascii=False)
    card("🎟️ Your Premiere ticket", "Copy the whole box below and paste it in the class Premiere thread on Canvas. "
         "The professor renders the top tickets as real AI video, and the class votes."
         f"<textarea style='width:100%;height:110px;font-family:monospace;font-size:11px;margin-top:8px' readonly>{esc(line)}</textarea>", GREEN)

## 💭 Before the Premiere
1. **Which layer caught the most?** A 🧱 code wall or the 📜 Brand Guardian? Which would you trust with a real client's money?
2. **Name the reason** for each upgrade you added: permissions, specialization, isolation or parallelism. Any upgrade without a reason?
3. **Your HW #9 pitch:** one rule written down, one check in code, one human gate. Which of today's three are you keeping?